# FULL-SYN — smoke và benchmark trên máy GPU (F0.7)

[Plan FULL-SYN](../../docs/project/full-syn-train-plan-2026-10-22.md), mục 4 (F0.7) và 4.1. Chạy **một lần trên mỗi máy mới**,
trước notebook `05_full_syn_r<N>`. Kết quả nằm trong `smoke/` của campaign, tách khỏi run thật.

Notebook kiểm:
1. Gói dữ liệu: mọi ảnh mà file nhãn liệt kê đều có; 300 dòng ngẫu nhiên của `SHA256SUMS` vẫn khớp.
2. FSA chạy `smoke_steps` bước, dừng, rồi chạy tiếp từ checkpoint thêm chừng ấy bước (kiểm resume).
3. PDP task 1 chạy `smoke_steps` bước từ detector COCO (như B2).
4. Giây/bước, VRAM tối đa, ước lượng giờ train của một vòng (chưa tính val mỗi epoch).

Xem kết quả rồi mới chạy notebook 05.

In [1]:
import json, os, sys
from pathlib import Path
REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'pyproject.toml').is_file())
sys.path.insert(0, str(REPO))
from autocheckout import full_syn
from autocheckout.io import load_json
from autocheckout.training_workflow import resolve
SETTINGS_PATH = resolve(os.environ.get('AUTOCHECKOUT_TRAINING_SETTINGS', 'data/training/settings_full_syn_r1.json'))
SETTINGS = load_json(SETTINGS_PATH)
PLAN = full_syn.prepare(SETTINGS)
print('Settings:', SETTINGS_PATH)
print('Campaign:', PLAN['root'], '| round', PLAN['round'], '| dry run' if PLAN['dry_run'] else '| real run')
print('Synthetic config sha256:', PLAN['release_config_sha256'])
print('Train images per task:', PLAN['train_images'])
print(json.dumps({k: PLAN[k] for k in ('resolution', 'batch_size', 'effective_batch', 'fsa_batch_size', 'num_workers', 'tf32',
                                         'fsa_epochs', 'epochs', 'audit_images')}, indent=1))

Settings: /workspace/AutoCheckout-CL/data/training/full-syn-r1/notebook_executions/04_full_syn_smoke/9406dc8a-a74a-441d-b854-8c37e5c68732/settings.json
Campaign: /workspace/AutoCheckout-CL/data/training/full-syn-r1 | round 1 | real run
Synthetic config sha256: cbd29ff0b63eb8aa93fdc48aef6d8eef0ca7116ac7cdc67bc3531fa6c5896eca
Train images per task: [21752, 12534, 12129, 11850, 13377]
{
 "resolution": 640,
 "batch_size": 8,
 "effective_batch": 8,
 "fsa_batch_size": 4,
 "num_workers": 8,
 "tf32": true,
 "fsa_epochs": 8,
 "epochs": 16,
 "audit_images": 1000
}


## 1. Máy và môi trường

In [2]:
import platform, subprocess, torch
from autocheckout.runinfo import git_state
git = git_state(REPO)
print('git', git.get('commit'), '| dirty' if git.get('dirty') else '| clean')
print('Python', platform.python_version(), '| torch', torch.__version__, '| CUDA', torch.version.cuda)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')
print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv'],
                     capture_output=True, text=True).stdout if torch.cuda.is_available() else '')

git e2dec6024f0093828fcceae56700736b29c5f481 | dirty
Python 3.10.22 | torch 2.2.2+cu121 | CUDA 12.1
GPU: NVIDIA GeForce RTX 4090
name, memory.total [MiB], driver_version
NVIDIA GeForce RTX 4090, 24564 MiB, 550.144.03



## 2. Smoke, resume và benchmark

In [3]:
SMOKE = full_syn.stage_smoke(PLAN, steps=SETTINGS.get('smoke_steps', 200))
print(json.dumps(SMOKE, indent=1))

2026-10-26T12:29:10+07:00 START bash /workspace/AutoCheckout-CL/scripts/run_exp.sh /workspace/AutoCheckout-CL/data/training/full-syn-r1/smoke/EXP-FULL-SYN-FSA/resolved_config.sh -> /workspace/AutoCheckout-CL/data/training/full-syn-r1/attempts/smoke/8ae200c2-aeba-425d-b0e0-a21b101aee7e/EXP-FULL-SYN-FSA_task1-1_val_runner.log
== EXP-FULL-SYN-FSA: 1 tasks in /workspace/AutoCheckout-CL/data/training/full-syn-r1/smoke/EXP-FULL-SYN-FSA (2026-10-26 05:29:10)
-- task 1: training (2026-10-26 05:29:10)
/venv/pdp/lib/python3.10/site-packages/lightning_utilities/core/imports.py:14: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
Seed set to 0
Multi-scale deformable attention: CUDA kernel
loading annotations into memory...
Done (t=0.15s)
creating index...
index created!
loading annotat

## 3. Đọc kết quả

- `data.missing_images` phải toàn 0 và `sample_mismatched` = 0.
- `fsa.resume_verification.state_equal` phải là `true`.
- So `training_hours_estimate` với mục 7 của plan (FSA 6–9 giờ, task 1 khoảng 10 giờ, task 2–5 khoảng 33 giờ).
  Lệch nhiều thì cập nhật ước lượng trước khi chạy notebook 05.
- VRAM tối đa phải còn dư so với bộ nhớ GPU.